# 09 · Webhooks — be told when a job finishes

**Use case.** A nightly pipeline re-uploads data and cleans; instead of polling, the orchestrator is told when the clean finishes (`job.succeeded`) and kicks off the next step. The delivery is signed, retried, and logged.

**What you will learn**
1. Register an https endpoint (a throw-away https://webhook.site URL here) for `clean` jobs
2. Send a signed test ping and verify the signature the way a receiver would
3. Run a clean and watch `job.succeeded` arrive exactly once
4. Read the delivery log; rotate the secret; delete the webhook
5. Receiver code you can paste into FastAPI or Express

**What this costs.** no credits; the clean is free. Needs the `webhooks:read` / `webhooks:write` scopes on the key.

> Every cell below ran for real against `api.langsat.ai` — the outputs are what the API returned. Re-running is safe:
> projects are found by name and reused, and a finished model is not retrained.

```mermaid
sequenceDiagram
    participant You
    participant Langsat
    participant Endpoint
    You->>Langsat: POST /webhooks {url, events, job_kinds}
    Langsat-->>You: {id, secret}  (secret shown once)
    You->>Langsat: POST /projects/{id}/clean
    Langsat->>Endpoint: POST job.succeeded  X-Langsat-Signature: t=…,v1=HMAC
    Endpoint-->>Langsat: 2xx within 10 s
    Note over Langsat,Endpoint: on failure: retry +1m, +5m, +30m, +2h · then exhausted
```

Set `WEBHOOK_URL` (and `WEBHOOK_SITE_TOKEN` — the uuid in the URL — so the notebook can read what arrived) before running; without them the live cells explain and skip.

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show, fig, metrics_table, project_models
from langsat import viz
import pandas as pd

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST-SDK-4' with 20 scopes


In [2]:
import os, time, urllib.request
from langsat.webhooks import verify_signature
from langsat import errors

URL = os.environ.get("WEBHOOK_URL")
TOKEN = os.environ.get("WEBHOOK_SITE_TOKEN")
if not URL:
    print("WEBHOOK_URL not set — open https://webhook.site, copy 'Your unique URL', and set WEBHOOK_URL / WEBHOOK_SITE_TOKEN")
p = get_or_create_project(ls, "explore", kind="data_analysis")
try:
    print("webhooks enabled:", ls.webhooks.enabled()["enabled"], "· key scopes:", [s for s in ls.me()["api_key"]["scopes"] if s.startswith("webhooks")])
    ls.webhooks.list()
except errors.MissingScope as e:
    print(f"this key cannot manage webhooks ({e.scope} missing) — mint one with the Full SDK preset; skipping the live steps")
    URL = None

reusing project 5c40e337-9753-4618-a7b3-574419f93c1e (amazon-reviews-explore, status=schema_done)
files already uploaded: ['customer.csv', 'product.csv', 'review.csv'] — skipping the upload
schema already detected — skipping
already cleaned — skipping
project ready: status=schema_done · type=data_analysis · files=['customer.csv', 'product.csv', 'review.csv']


webhooks enabled: True · key scopes: ['webhooks:read', 'webhooks:write']


## Register and test

The secret is returned **once**. `test()` sends a `ping` immediately; reading it back from webhook.site and verifying with `verify_signature` is exactly what your receiver will do.

In [3]:
def received():
    with urllib.request.urlopen(f"https://webhook.site/token/{TOKEN}/requests?sorting=newest", timeout=20) as r:
        return json.load(r)["data"]

if URL:
    for w in ls.webhooks.list():
        if (w.get("description") or "") == "amazon-reviews example":
            ls.webhooks.delete(w["id"])
    w = ls.webhooks.create(URL, job_kinds=["clean"], description="amazon-reviews example")
    secret = w["secret"]                       # shown once
    print("webhook", w["id"], "events", w["events"], "kinds", w["job_kinds"])
    t = ls.webhooks.test(w["id"])
    print("ping delivered:", t["ok"], "· HTTP", t["delivery"]["last_status_code"])
    if TOKEN:
        time.sleep(2)
        hit = next(x for x in received() if x["headers"].get("x-langsat-event") == ["ping"])
        ev = verify_signature(hit["content"], hit["headers"]["x-langsat-signature"][0], secret=secret)
        print("signature verified · event", ev["id"], ev["type"])
        print("headers the receiver saw:", {k: v[0] for k, v in hit["headers"].items() if k.startswith("x-langsat")})

webhook b3374cfa-42be-4751-942e-0e3f652c16d8 events ['job.failed', 'job.succeeded'] kinds ['clean']


ping delivered: True · HTTP 200


signature verified · event evt_ping_2833b60cf0f54b39 ping
headers the receiver saw: {'x-langsat-signature': 't=1789527969,v1=09ca9eb9dcd871a29c43093a4bdebfccd6d847159929e957d082f0def167ce16', 'x-langsat-delivery': 'whd_31335b807323bf92d969', 'x-langsat-event': 'ping'}


## A real job

A clean on the explore project. The event names the job and where to fetch its result (`result_route`) — never the data itself.

In [4]:
if URL:
    p.cleaning.clean().wait(timeout=1800)
    print("clean finished; waiting for the webhook …")
    got = None
    if TOKEN:
        for _ in range(24):
            time.sleep(5)
            got = next((x for x in received() if x["headers"].get("x-langsat-event") == ["job.succeeded"]), None)
            if got: break
        if got:
            ev = verify_signature(got["content"], got["headers"]["x-langsat-signature"][0], secret=secret)
            job = ev["data"]["job"]
            print("job.succeeded ·", job["kind"], job["status"], "· project", ev["data"]["project_id"] == p.id, "· fetch the result at", job["result_route"])
            mine = {d["delivery_id"] for d in ls.webhooks.deliveries(w["id"])}       # this webhook's deliveries only
            n = sum(1 for x in received() if x["headers"].get("x-langsat-event") == ["job.succeeded"]
                    and x["headers"].get("x-langsat-delivery", [""])[0] in mine)
            print("job.succeeded deliveries from this webhook that reached the endpoint:", n, "(exactly once)")
    print(pd.DataFrame(ls.webhooks.deliveries(w["id"]))[["event", "status", "attempts", "last_status_code", "created_at"]])

clean finished; waiting for the webhook …


job.succeeded · clean succeeded · project True · fetch the result at /api/v1/projects/5c40e337-9753-4618-a7b3-574419f93c1e/status


job.succeeded deliveries from this webhook that reached the endpoint: 1 (exactly once)


           event     status  attempts  last_status_code  \
0  job.succeeded  succeeded         1               200   
1           ping  succeeded         1               200   

                         created_at  
0  2026-09-16T03:06:20.103653+00:00  
1  2026-09-16T03:06:09.494318+00:00  


## Receiver code

Use the **raw** request bytes — a re-serialised body will not match the signature.

```python
# FastAPI
from fastapi import FastAPI, Request, HTTPException
from langsat.webhooks import verify_signature, InvalidSignature
app = FastAPI()

@app.post("/langsat")
async def hook(request: Request):
    try:
        event = verify_signature(await request.body(), request.headers.get("X-Langsat-Signature"), secret=SECRET)
    except InvalidSignature:
        raise HTTPException(400)
    if event["type"] == "job.succeeded" and event["data"]["job"]["kind"] == "clean":
        start_next_step(event["data"]["project_id"])
    return {"ok": True}
```

```ts
// Express
import express from "express";
import { verifySignature } from "@langsat/sdk";
const app = express();
app.post("/langsat", express.raw({ type: "*/*" }), async (req, res) => {
  try {
    const event = await verifySignature(req.body, req.header("x-langsat-signature"), { secret: process.env.LANGSAT_WEBHOOK_SECRET! });
    if (event.type === "job.succeeded") await startNextStep(event.data.project_id);
    res.sendStatus(200);
  } catch { res.sendStatus(400); }
});
```

In [5]:
if URL:
    rotated = ls.webhooks.rotate_secret(w["id"])
    print("rotated:", rotated["secret"] != secret)
    ls.webhooks.delete(w["id"]); print("deleted")
save_metrics(".", {"notebook": "09_webhooks", "task": "webhooks · job.succeeded on clean", "model": "—", "project_id": p.id,
                   "headline": {"ran_live": bool(URL), "ping_ok": (t["ok"] if URL else None), "job_succeeded_seen": (bool(got) if URL and TOKEN else None)}})

rotated: True


deleted
wrote results/metrics.json


PosixPath('results/metrics.json')

## Where to go next

- Docs: https://langsat.ai/resources/learn/getting-started/sdk#webhooks
- [10 · Credits & errors](../10_credits_estimates_and_errors/) — the error codes a receiver-side script can branch on